# 01a — PE-Core-G14-448 Feature Extraction (Colab A100 80GB high-RAM)

A100-80GB-optimized version. Run `00_manifest_qc.ipynb` first (requires
`val_split.parquet` + `val_zeroshot_scene.parquet` for stage val encoding).

**Outputs:**

- `features/pe_g14/train/chunk_*.npz` — train image embeddings (chunked, resumable)
- `features/pe_g14/train_text/chunk_*.npz` — train caption embeddings
- `features/pe_g14/gallery.npz` — gallery image embeddings (1978, 1280) fp16
- `features/pe_g14/queries.npz` — query text embeddings (1978, 1280) fp16
- `features/pe_g14/val.npz` + `val_text.npz` — val_split image + text emb
- `features/pe_g14/val_zs.npz` + `val_zs_text.npz` — val_zeroshot_scene image + text emb

**A100-80GB tuning:**

- IMAGE_BATCH_SIZE 96 → 256, TEXT_BATCH_SIZE 256 → 512
- TRAIN_CHUNK_SIZE 4096 → 8192 (fewer chunks, reduced sync overhead)
- NUM_WORKERS 8 → 12, BF16 autocast + channels_last + torch.compile
- Flash SDPA + TF32 matmul
- Resume-safe: checks chunks on both local and Drive before encoding


In [ ]:
# === Bootstrap & config ===
from pathlib import Path
import gc
import os
import subprocess
import sys
import warnings

warnings.filterwarnings('ignore')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

# aic_colab_utils.py must be in the same directory as the notebook on Colab.
# On Colab: !cp /content/drive/MyDrive/aic2026_data/code/aic_colab_utils.py .
# Or upload directly to /content/.
NOTEBOOK_DIR = Path('.').resolve()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from aic_colab_utils import (
    mirror_dataset_to_drive,
    mirror_raw_as_tar_split,        # NEW: tar+split raw → Drive (10× faster than rsync many-files)
    setup_aic2026_environment,
    select_a100_device,
    sync_chunk_to_drive,
    wait_for_pending_syncs,
    find_existing_chunks,
    chunk_done,
    l2_normalize_np,
    save_npz_atomic,
    chunk_file,
    maybe_clear_cuda,
)

PATHS = setup_aic2026_environment(
    # kaggle_token_path='/content/drive/MyDrive/aic2026_data/.kaggle/kaggle.json',
    # force_redownload=False,
)
INPUT_ROOT = PATHS['input_root']
MANIFEST_DIR = PATHS['manifest_dir']
OUTPUT_ROOT = PATHS['output_root']            # /content/aic_local/output
DRIVE_OUTPUT_ROOT = PATHS['drive_output_root']  # /content/drive/.../output
LOCAL_ROOT = PATHS['local_root']
FEATURE_DIR = OUTPUT_ROOT / 'features' / 'pe_g14'
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

# --- Run knobs ---
TRAIN_START_IDX = 0
TRAIN_END_IDX = None

SMOKE_TEST = False
SMOKE_N_TRAIN = 100
SMOKE_N_GALLERY = 100

# Train embeddings are not consumed by any notebook in the pipeline →
# save ~5h by disabling by default. Re-enable if needed for pose prototype
# synthesis or auxiliary loss later.
RUN_TRAIN_IMAGE_EMBEDDINGS = False    # ← disabled: train embeddings unused
RUN_TRAIN_TEXT_EMBEDDINGS = False     # ← disabled: train_text embeddings unused
RUN_GALLERY_IMAGE_EMBEDDINGS = True   # ← required for notebook 07 Round 4 PE-G14
RUN_QUERY_TEXT_EMBEDDINGS = True      # ← required for notebook 07 Round 4 PE-G14
RUN_VAL_IMAGE_EMBEDDINGS = True       # ← for 03 fallback compare
RUN_VAL_TEXT_EMBEDDINGS = True

# --- Drive persistence strategy ---
# False: skip mirror raw (Kaggle re-DL ~2h per session — NOT recommended)
# 'tar': tar+split raw → Drive (~30-40 min one-time, restore ~30-40 min)
# 'rsync': rsync many-files (~3-10h, SLOW, legacy)
MIRROR_RAW_STRATEGY = 'tar'

# --- A100-80GB tuned hyperparams ---
# Increased from A100-40GB defaults (96/256/4096) to take advantage of 80GB high-RAM.
PE_MODEL_NAME = 'PE-Core-G14-448'
IMAGE_BATCH_SIZE = 256         # A100-40GB: 96; A100-80GB: 256 (PE-G14 1.8B fits w/ BF16+channels_last)
TEXT_BATCH_SIZE = 512          # A100-40GB: 256
TRAIN_CHUNK_SIZE = 8192        # A100-40GB: 4096 (fewer chunks → reduced sync overhead)
MIN_IMAGE_BATCH_SIZE = 16      # OOM fallback floor
NUM_WORKERS = 12               # high-RAM allows 12 workers
PREFETCH_FACTOR = 4
USE_BF16 = True                # A100 native BF16
# torch.compile mode options:
#   False             — disable compile (safest, A100 BF16+Flash-SDPA already very fast)
#   'default'         — kernel fusion, NO CUDA Graphs (safe)
#   'reduce-overhead' — CUDA Graphs, FASTEST but conflicts with PE-G14 RoPE cache
#                       (error: "accessing tensor output of CUDAGraphs that has been overwritten")
#   'max-autotune'    — best perf but slow to compile on first run
USE_COMPILE = False            # ← disabled by default to avoid CUDA Graphs bug with PE-G14
CHANNELS_LAST = True
CLEAR_CACHE_EVERY_N_IMAGE_BATCHES = 400  # 80GB does not need aggressive cache clearing
CLEAR_CACHE_EVERY_N_TEXT_BATCHES = 400
ASYNC_DRIVE_SYNC = True

print('OUTPUT_ROOT:', OUTPUT_ROOT)
print('DRIVE_OUTPUT_ROOT:', DRIVE_OUTPUT_ROOT)
print('FEATURE_DIR:', FEATURE_DIR)
print(f'MIRROR_RAW_STRATEGY = {MIRROR_RAW_STRATEGY!r}')
print(f'USE_COMPILE = {USE_COMPILE!r}')


In [ ]:
# === Install deps (only run on first launch / new runtime) ===
def pip_install(*packages):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])

try:
    import core.vision_encoder.pe as pe
    import core.vision_encoder.transforms as pe_transforms
except Exception:
    repo = NOTEBOOK_DIR / 'perception_models'
    if not repo.exists():
        subprocess.check_call([
            'git', 'clone',
            'https://github.com/facebookresearch/perception_models.git',
            str(repo),
        ])
    pip_install('-e', str(repo))
    if str(repo) not in sys.path:
        sys.path.append(str(repo))
    import core.vision_encoder.pe as pe
    import core.vision_encoder.transforms as pe_transforms

try:
    import transformers  # noqa: F401
except Exception:
    pip_install('transformers>=4.52.3', 'accelerate', 'timm', 'einops', 'safetensors')

import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm.auto import tqdm


In [ ]:
# === Device + manifest load ===
device = select_a100_device()
amp_dtype = torch.bfloat16 if (USE_BF16 and device.type == 'cuda') else torch.float16
memory_format = torch.channels_last if (CHANNELS_LAST and device.type == 'cuda') else torch.contiguous_format

train_df = pd.read_parquet(MANIFEST_DIR / 'train_manifest.parquet')
gallery_df = pd.read_parquet(MANIFEST_DIR / 'gallery_manifest.parquet')
query_df = pd.read_parquet(MANIFEST_DIR / 'query_manifest.parquet')

# Manifest from Kaggle dataset may contain a Kaggle-prefix path.
# Map to local SSD by replacing /kaggle/input/datasets/... → INPUT_ROOT.
def remap_path(p):
    if not isinstance(p, str) or not p:
        return p
    if p.startswith(str(INPUT_ROOT)):
        return p
    # Find trail from a known anchor directory.
    for anchor in ('annotation/', 'train/imgs_', 'name-masked_test-set/', 'gallery/'):
        i = p.find(anchor)
        if i >= 0:
            return str(INPUT_ROOT / p[i:])
    return p

for df_ in (train_df, gallery_df):
    if 'image_path' in df_.columns:
        df_['image_path'] = df_['image_path'].map(remap_path)

if SMOKE_TEST:
    train_df = train_df.head(SMOKE_N_TRAIN).copy()
    gallery_df = gallery_df.head(SMOKE_N_GALLERY).copy()

train_df = train_df[train_df['image_path'].astype(str).str.len() > 0].reset_index(drop=True)
if TRAIN_END_IDX is not None:
    train_df = train_df.iloc[TRAIN_START_IDX:TRAIN_END_IDX].reset_index(drop=True)
else:
    train_df = train_df.iloc[TRAIN_START_IDX:].reset_index(drop=True)

print('train:', train_df.shape, '| gallery:', gallery_df.shape, '| query:', query_df.shape)


In [ ]:
# === Load PE-Core-G14-448 with channels_last + (optional) torch.compile ===
maybe_clear_cuda()
print('Loading', PE_MODEL_NAME)
pe_model = pe.CLIP.from_config(PE_MODEL_NAME, pretrained=True).to(device).eval()
if memory_format == torch.channels_last:
    pe_model = pe_model.to(memory_format=torch.channels_last)
pe_preprocess = pe_transforms.get_image_transform(pe_model.image_size)
pe_tokenizer = pe_transforms.get_text_tokenizer(pe_model.context_length)
_fallback_size = pe_model.image_size if isinstance(pe_model.image_size, int) else pe_model.image_size[0]
pe_fallback_tensor = pe_preprocess(Image.new('RGB', (_fallback_size, _fallback_size))).clone()
print('image_size:', pe_model.image_size, '| context_length:', pe_model.context_length)

# torch.compile setup — polymorphic based on USE_COMPILE
#   False             → skip compile (recommended, safe)
#   'default'         → kernel fusion, no CUDA Graphs
#   'reduce-overhead' → CUDA Graphs (BUG with PE-G14 RoPE cache mutation)
#   'max-autotune'    → best perf, compile slow
if USE_COMPILE and device.type == 'cuda':
    _compile_mode = 'default' if USE_COMPILE is True else USE_COMPILE
    if _compile_mode == 'reduce-overhead':
        print(f"⚠️  USE_COMPILE='reduce-overhead' may conflict with the PE-G14 RoPE cache. "
              f"If you see 'accessing tensor output of CUDAGraphs', set USE_COMPILE='default' or False.")
    try:
        pe_model.encode_image = torch.compile(
            pe_model.encode_image, mode=_compile_mode, dynamic=False, fullgraph=False,
        )
        pe_model.encode_text = torch.compile(
            pe_model.encode_text, mode=_compile_mode, dynamic=True, fullgraph=False,
        )
        print(f'torch.compile mode={_compile_mode!r} enabled for encode_image / encode_text')
    except Exception as exc:
        print('torch.compile failed — fallback eager:', exc)
else:
    print('torch.compile disabled (USE_COMPILE=False) — A100 BF16+Flash-SDPA is already fast enough.')


In [ ]:
# === DataLoader + encode helpers ===
class PEImageDataset(Dataset):
    def __init__(self, df, id_col, preprocess, fallback_tensor):
        self.paths = df['image_path'].astype(str).tolist()
        self.ids = df[id_col].astype(str).tolist()
        self.preprocess = preprocess
        self.fallback = fallback_tensor

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        path = self.paths[i]
        try:
            with Image.open(path) as img:
                tensor = self.preprocess(img.convert('RGB'))
            return tensor, self.ids[i], path, True
        except Exception:
            return self.fallback.clone(), self.ids[i], path, False


def _make_loader(dataset, batch_size, workers):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        num_workers=workers,
        pin_memory=(device.type == 'cuda'),
        persistent_workers=(workers > 0),
        prefetch_factor=PREFETCH_FACTOR if workers > 0 else None,
        shuffle=False,
        drop_last=False,
    )


@torch.inference_mode()
def encode_image_df_to_npz(df, id_col, out_path, batch_size=IMAGE_BATCH_SIZE):
    out_path = Path(out_path)
    if chunk_done(out_path, DRIVE_OUTPUT_ROOT, LOCAL_ROOT):
        print('Skip existing', out_path.name)
        return
    if len(df) == 0:
        return
    df = df.reset_index(drop=True)

    current_bs = int(batch_size)
    all_ids, all_paths, all_feats, all_ok = [], [], [], []

    while True:
        try:
            loader = _make_loader(
                PEImageDataset(df, id_col, pe_preprocess, pe_fallback_tensor),
                batch_size=current_bs, workers=NUM_WORKERS,
            )
            pbar = tqdm(total=len(df), desc=f'PE img → {out_path.name}', unit='img')
            all_ids.clear(); all_paths.clear(); all_feats.clear(); all_ok.clear()
            batch_count = 0
            for tensors, ids, paths, ok in loader:
                tensors = tensors.to(device, non_blocking=True)
                if memory_format == torch.channels_last:
                    tensors = tensors.contiguous(memory_format=torch.channels_last)
                with torch.autocast(device_type=device.type, dtype=amp_dtype,
                                    enabled=(device.type == 'cuda')):
                    feats = pe_model.encode_image(tensors)
                feats = F.normalize(feats.float(), dim=-1).cpu().numpy()
                all_ids.extend(list(ids))
                all_paths.extend(list(paths))
                all_feats.append(feats)
                all_ok.append(np.asarray(ok, dtype=bool))
                pbar.update(len(ids))
                pbar.set_postfix(bs=current_bs)
                batch_count += 1
                if batch_count % CLEAR_CACHE_EVERY_N_IMAGE_BATCHES == 0:
                    maybe_clear_cuda()
            pbar.close()
            break
        except torch.cuda.OutOfMemoryError:
            try:
                pbar.close()
            except Exception:
                pass
            maybe_clear_cuda()
            if current_bs <= MIN_IMAGE_BATCH_SIZE:
                raise
            current_bs = max(MIN_IMAGE_BATCH_SIZE, current_bs // 2)
            print(f'CUDA OOM → retrying with batch_size={current_bs}')

    emb = l2_normalize_np(np.concatenate(all_feats, axis=0)).astype('float16')
    ok_arr = np.concatenate(all_ok, axis=0)
    assert len(emb) == len(df), (len(emb), len(df))
    assert np.isfinite(emb).all()
    save_npz_atomic(
        out_path,
        ids=np.array(all_ids), paths=np.array(all_paths),
        embeddings=emb, ok=ok_arr,
    )
    if ASYNC_DRIVE_SYNC:
        sync_chunk_to_drive(out_path, LOCAL_ROOT, DRIVE_OUTPUT_ROOT, background=True)
    maybe_clear_cuda()
    print('Wrote', out_path, emb.shape, 'bad:', int((~ok_arr).sum()))


@torch.inference_mode()
def encode_texts_to_npz(ids, texts, out_path, batch_size=TEXT_BATCH_SIZE):
    out_path = Path(out_path)
    if chunk_done(out_path, DRIVE_OUTPUT_ROOT, LOCAL_ROOT):
        print('Skip existing', out_path.name)
        return
    feats = []
    pbar = tqdm(range(0, len(texts), batch_size), desc=f'PE txt → {out_path.name}')
    for step, start in enumerate(pbar, start=1):
        batch = list(texts[start:start + batch_size])
        tokens = pe_tokenizer(batch).to(device)
        with torch.autocast(device_type=device.type, dtype=amp_dtype,
                            enabled=(device.type == 'cuda')):
            tf = pe_model.encode_text(tokens)
        feats.append(F.normalize(tf.float(), dim=-1).cpu().numpy())
        if step % CLEAR_CACHE_EVERY_N_TEXT_BATCHES == 0:
            maybe_clear_cuda()
    emb = l2_normalize_np(np.concatenate(feats, axis=0)).astype('float16')
    assert np.isfinite(emb).all()
    save_npz_atomic(out_path, ids=np.array(ids), embeddings=emb)
    if ASYNC_DRIVE_SYNC:
        sync_chunk_to_drive(out_path, LOCAL_ROOT, DRIVE_OUTPUT_ROOT, background=True)
    maybe_clear_cuda()
    print('Wrote', out_path, emb.shape)


def encode_train_images_chunked():
    out_dir = FEATURE_DIR / 'train'
    out_dir.mkdir(parents=True, exist_ok=True)
    for start in range(0, len(train_df), TRAIN_CHUNK_SIZE):
        end = min(start + TRAIN_CHUNK_SIZE, len(train_df))
        out_path = chunk_file(out_dir, start, end)
        encode_image_df_to_npz(train_df.iloc[start:end], 'image_id', out_path)


def encode_train_text_chunked():
    out_dir = FEATURE_DIR / 'train_text'
    out_dir.mkdir(parents=True, exist_ok=True)
    for start in range(0, len(train_df), TRAIN_CHUNK_SIZE):
        end = min(start + TRAIN_CHUNK_SIZE, len(train_df))
        out_path = chunk_file(out_dir, start, end)
        part = train_df.iloc[start:end]
        encode_texts_to_npz(
            part['image_id'].astype(str).tolist(),
            part['caption'].astype(str).tolist(),
            out_path,
        )


In [ ]:
# === Run encoding stages ===
if RUN_TRAIN_IMAGE_EMBEDDINGS:
    encode_train_images_chunked()

if RUN_TRAIN_TEXT_EMBEDDINGS:
    encode_train_text_chunked()

if RUN_GALLERY_IMAGE_EMBEDDINGS:
    encode_image_df_to_npz(gallery_df, 'gallery_id', FEATURE_DIR / 'gallery.npz')

if RUN_QUERY_TEXT_EMBEDDINGS:
    encode_texts_to_npz(
        query_df['query_index'].astype(str).tolist(),
        query_df['caption'].astype(str).tolist(),
        FEATURE_DIR / 'queries.npz',
    )

# === Val splits encoding (for adaptive ensemble + ablation in 09) ===
# Encode image + text for val_split + val_zeroshot_scene to use as
# val set when tuning adaptive weighting and benchmarking per-model val mAP.
if RUN_VAL_IMAGE_EMBEDDINGS or RUN_VAL_TEXT_EMBEDDINGS:
    for _split_name in ('val_split', 'val_zeroshot_scene'):
        _split_path = MANIFEST_DIR / f'{_split_name}.parquet'
        if not _split_path.exists():
            print(f'⚠️  {_split_path.name} not found — re-run 00_manifest_qc to produce it. Skip.')
            continue
        _df = pd.read_parquet(_split_path)
        if 'image_path' in _df.columns:
            _df['image_path'] = _df['image_path'].map(remap_path)
        _df = _df[_df['image_path'].astype(str).str.len() > 0].reset_index(drop=True)
        if len(_df) == 0:
            print(f'⚠️  {_split_name} is empty — skip.')
            continue
        _short = 'val' if _split_name == 'val_split' else 'val_zs'
        if RUN_VAL_IMAGE_EMBEDDINGS:
            encode_image_df_to_npz(_df, 'image_id', FEATURE_DIR / f'{_short}.npz')
        if RUN_VAL_TEXT_EMBEDDINGS:
            encode_texts_to_npz(
                _df['image_id'].astype(str).tolist(),
                _df['caption'].astype(str).tolist(),
                FEATURE_DIR / f'{_short}_text.npz',
            )

wait_for_pending_syncs()

# === Mirror raw + manifests to Drive (persistent cache) ===
#
# Manifests ~400MB / ~10 files: rsync nhanh (~1-2 min).
# Raw ~100GB / ~1M files: select strategy based on MIRROR_RAW_STRATEGY:
#   - 'tar':   tar+split → Drive parts <5GB. One-time ~30-40 min.
#              Session sau restore_raw_from_tar_split() ~30-40 min (vs Kaggle re-DL ~2h).
#   - 'rsync': many-small-files rsync. SLOW (~3-10h), not recommended.
#   - False:   skip raw mirror; session sau Kaggle re-DL ~2h.
if MIRROR_RAW_STRATEGY == 'tar':
    mirror_raw_as_tar_split(PATHS)                                              # raw → tar parts
    mirror_dataset_to_drive(PATHS, include_raw=False, include_manifests=True)   # manifests only
elif MIRROR_RAW_STRATEGY == 'rsync':
    mirror_dataset_to_drive(PATHS, include_raw=True, include_manifests=True)
else:
    mirror_dataset_to_drive(PATHS, include_raw=False, include_manifests=True)

for _name in ['pe_model', 'pe_preprocess', 'pe_tokenizer', 'pe_fallback_tensor']:
    globals().pop(_name, None)
maybe_clear_cuda()
print('Released PE model. Done.')
